# of how many DEP is the Head-ID higher than the Token-ID

In [1]:
#first parse the corpus
# Import the toolkit, giving it a shorter alias 'mptf'
import mptf_parser as mptf

# Set your folder paths
input_folder = r"C:\Users\rahaa\Dropbox\MPCD\exports_28-1-2026"
output_folder = r"C:\Users\rahaa\Dropbox\MPCD\the syntax project\nounphrase\export_files\conllu_output"

In [2]:
import os
import mptf_parser as mptf

# We need to import the 'parse' function and field parsers that your module uses internally
from conllu import parse
from conllu.parser import DEFAULT_FIELD_PARSERS

# --- STEP 1: Define Genre Mapping ---
genre_mapping = {
    "AfZ-TD2-01": "zand", "AOD-K20-01": "andarz", "Col-J2-1-J2-01": "colophon",
    "Col-K1-1-K1-01": "colophon", "Col-K5-1-K5-01": "colophon", "Col-K5-2-K5-01": "colophon",
    "Col-K7b-1-K7b-01": "colophon", "Col-K7b-2-K7b-01": "colophon", "Col-Pt4-1-Pt4-01": "colophon",
    "Col_RFS_TD2-TD2-01": "colophon", "DD-K35-01": "theological", "DD-TD4a-01": "theological",
    "DH_Col1-DH-DH-01": "colophon", "Dk3-B-01": "theological", "Dk4-B-01": "theological",
    "Dk5-B-01": "theological", "Dk5-test": "theological", "Dk6-B-01": "andarz",
    "Dk7-B-01": "theological", "Dk8-B-01": "theological", "Dk9-B-01": "theological",
    "DMX-K43a-01": "andarz", "DMX-L19-01": "andarz", "FO-K20-01": "zand", "FrW10-K20-01": "zand",
    "GA-K20-01": "andarz", "GBd-DH-01": "zand-based", "GBd-TD1-01": "zand-based",
    "GBd-TD2-01": "zand-based", "H-TD-01": "zand", "K20_Col1-K20-K20-01": "colophon",
    "K43a_Col2-K43a-01": "colophon", "MHD-MHDC-01": "juridical", "MYF-K20-01": "andarz",
    "MYF-K26-01": "andarz", "N-TD-01": "zand", "NAA-K7b-01": "ritual", "NAP-4050-01": "ritual",
    "Ner-K43c-01": "ritual", "NM-K35-01": "theological", "NM-TD4a-01": "theological",
    "NM_K35": "theological", "NM_TD4a": "theological", "NPS-TD2-01": "ritual",
    "P-TD2-01": "zand-based", "PHN-K20-01": "theological", "PNN-K7b-01": "andarz",
    "PV-K1-02": "zand", "PVr-K7a-01": "zand", "PVyt-F12a-01": "zand", "PY-Pt4-01": "zand",
    "Pyextr-K7a-01": "zand", "RAF-TD2-01": "juridical", "RFS-TD2-01": "juridical",
    "RThQA-TD2-01": "juridical", "SnS-K20-01": "zand-based", "TD1_Col1-TD1-TD1-01": "colophon",
    "TD2_Col1-TD2-TD1-01": "colophon", "WZ-K35-01": "theological", "WZ_K35": "theological",
    "ZPJ-TD2-01": "zand-based", "ZWY-K20-01": "zand-based", "ZWY-K43a-01": "zand-based"
}

# --- STEP 2: Convert Source Files (if needed) ---
print("--- Starting Corpus Parsing Pipeline with Genre Info---")
print(f"Step 1: Converting files from '{input_folder}'")
mptf._process_directory(input_folder, output_folder)

# --- STEP 3: Load Corpus Manually to Add Genre ---
print(f"\nStep 2: Loading processed files from '{output_folder}' and adding genre.")

custom_field_parsers = DEFAULT_FIELD_PARSERS.copy()
custom_field_parsers['id'] = lambda line, i: line[i]
custom_field_parsers['head'] = lambda line, i: line[i]

all_corpus_sentences_list = []
conllu_files = [f for f in os.listdir(output_folder) if f.lower().endswith('.conllu')]

for filename in conllu_files:
    assigned_genre = "unknown"
    for key, genre in genre_mapping.items():
        if key in filename:
            assigned_genre = genre
            break
    
    file_path = os.path.join(output_folder, filename)
    with open(file_path, 'r', encoding='utf-8') as f:
        data = f.read()
    
    # --- FIX ADDED HERE ---
    # Re-introducing the cleaning step from your original parser.
    # This removes any invalid lines before parsing to prevent the error.
    lines = data.splitlines()
    clean_lines = [line for line in lines if line.startswith('#') or line.strip() == "" or line.count('\t') == 9]
    clean_data = "\n".join(clean_lines) + "\n"
    
    # Parse the CLEANED data instead of the raw data
    sentences_from_lib = parse(clean_data, field_parsers=custom_field_parsers)
    
    for lib_sentence in sentences_from_lib:
        new_sentence = mptf.Sentence(lib_sentence, source_filename=filename)
        new_sentence.metadata['genre'] = assigned_genre
        all_corpus_sentences_list.append(new_sentence)

my_corpus = all_corpus_sentences_list
print(f"\nSuccessfully created 'my_corpus' with {len(my_corpus)} sentences with genre information.")

# --- STEP 4: Filter the corpus (same as your original code) ---
syntactically_parsed_corpus = []
for sentence in my_corpus:
    if any(token.deprel == 'root' for token in sentence.get_tokens()):
        syntactically_parsed_corpus.append(sentence)



print(f"Original corpus size: {len(my_corpus)} sentences")
print(f"Filtered corpus size (with syntactic parse): {len(syntactically_parsed_corpus)} sentences")

# --- STEP 5: New, flexible search function ---
def search_corpus(corpus, search_term=None, genre=None, case_sensitive=False):
    matching_sentences = []
    query = search_term.lower() if search_term and not case_sensitive else search_term

    for sentence in corpus:
        if genre and sentence.metadata.get('genre') != genre:
            continue
        if not query:
            matching_sentences.append(sentence)
            continue
        for token in sentence.get_tokens():
            token_form = str(token.form) if token.form else ""
            comparison_form = token_form.lower() if not case_sensitive else token_form
            if query == comparison_form:
                matching_sentences.append(sentence)
                break
    return matching_sentences

# --- Example Searches ---
search_query = "dānāg"
target_genre = "andarz"
results = search_corpus(syntactically_parsed_corpus, search_term=search_query, genre=target_genre)
print(f"\nFound {len(results)} sentences containing '{search_query}' in the genre '{target_genre}'.")

results_all_genres = search_corpus(syntactically_parsed_corpus, search_term=search_query)
print(f"Found {len(results_all_genres)} sentences containing '{search_query}' across all genres.")

juridical_sentences = search_corpus(syntactically_parsed_corpus, genre="juridical")
print(f"Found {len(juridical_sentences)} sentences in the 'juridical' genre.")

--- Starting Corpus Parsing Pipeline with Genre Info---
Step 1: Converting files from 'C:\Users\rahaa\Dropbox\MPCD\exports_28-1-2026'


Converting source files:   0%|          | 0/46 [00:00<?, ?it/s]


Source file conversion complete!

Step 2: Loading processed files from 'C:\Users\rahaa\Dropbox\MPCD\the syntax project\nounphrase\export_files\conllu_output' and adding genre.

Successfully created 'my_corpus' with 39836 sentences with genre information.
Original corpus size: 39836 sentences
Filtered corpus size (with syntactic parse): 5275 sentences

Found 0 sentences containing 'dānāg' in the genre 'andarz'.
Found 32 sentences containing 'dānāg' across all genres.
Found 0 sentences in the 'juridical' genre.


In [3]:
def count_forward_conj(corpus):
    total_conj = 0
    forward_conj = 0
    examples = []

    for sentence in corpus:
        for token in sentence.get_tokens():
            if token.deprel == "conj":
                total_conj += 1

                # Skip if ID or HEAD is missing or non-numeric
                if not token.id or not token.head:
                    continue
                if not str(token.id).isdigit() or not str(token.head).isdigit():
                    continue

                token_id = int(token.id)
                head_id = int(token.head)

                if head_id > token_id:
                    forward_conj += 1
                    examples.append((sentence, token))

    return total_conj, forward_conj, examples


total_conj, forward_conj, examples = count_forward_conj(syntactically_parsed_corpus)

print(f"Total 'conj' tokens: {total_conj}")
print(f"'conj' tokens with head AFTER them: {forward_conj}")
print(f"Percentage: {forward_conj/total_conj*100:.2f}%")

Total 'conj' tokens: 6460
'conj' tokens with head AFTER them: 459
Percentage: 7.11%


In [13]:
for sent, tok in examples[:10]:
    
    sent_id = sent.metadata.get("SENTENCE ID", "UNKNOWN_SENT_ID")
    
    print(f"Sentence ID: {sent_id}")
    print(f"Token: {tok.form} (ID={tok.id}) → Head ID={tok.head}")
    
    # Get sentence text safely
    text = sent.metadata.get("text")
    if not text:
        text = " ".join(t.form for t in sent.get_tokens() if t.form)
    
    print(f"Text: {text}")
    print("-" * 60)

Sentence ID: AOD-K20 s45	_	_	_	_	_	_	_	_	_
Token: dušram (ID=4) → Head ID=9
Text: _ _ _ panǰom pad anāgīh dušram pad nēkīh mast nē bawēd ,
------------------------------------------------------------
Sentence ID: DD-K35 s20	_	_	_	_	_	_	_	_	_
Token: awištāb (ID=10) → Head ID=22
Text: _ _ _ _ ud az ān ī škeft kōxšišnīh ī druz rāy awištāb wizōyišnīg menišnīh _ ud az ān ī awizīrišnīg frēzwānīg X kār kam pardazišnīh ast
------------------------------------------------------------
Sentence ID: DD-K35 s28	_	_	_	_	_	_	_	_	_
Token: čand (ID=1) → Head ID=20
Text: _ _ _ _ čand u =m az dēnāgāhīh _ u =m pad ōš ayādīh _ , ud az pēšēnīgān dastwarīh _ ud pad xrad sahišn ast pāsox azēr ī pursišn nibišt kāmam
------------------------------------------------------------
Sentence ID: DD-K35 s28	_	_	_	_	_	_	_	_	_
Token: =m (ID=7) → Head ID=20
Text: _ _ _ _ čand u =m az dēnāgāhīh _ u =m pad ōš ayādīh _ , ud az pēšēnīgān dastwarīh _ ud pad xrad sahišn ast pāsox azēr ī pursišn nibišt kāmam
-------------------

In [25]:
# ============================================================
# DK5 forward-headed 'conj' tokens with propagated newpart
# ============================================================

dk5_forward = []

for sentence in syntactically_parsed_corpus:
    
    raw_sid = sentence.metadata.get("SENTENCE ID", "")
    clean_sid = raw_sid.split("\t")[0].strip()
    
    # Filter only DK5 sentences (case-insensitive)
    if not clean_sid.lower().startswith("dk5"):
        continue
    
    last_newpart = None  # will store the last non-empty newpart seen in this sentence
    
    for token in sentence.get_tokens():
        
        # Update last_newpart if this token has it
        token_newpart = token.misc.get("newpart")
        if token_newpart:
            last_newpart = token_newpart
        
        # Only keep forward-headed 'conj'
        if (
            token.deprel == "conj"
            and str(token.id).isdigit()
            and str(token.head).isdigit()
            and int(token.head) > int(token.id)
        ):
            # Append with current/new last_newpart
            dk5_forward.append((clean_sid, sentence, token, last_newpart))


# ============================================================
# Print all results
# ============================================================

print(f"Total DK5 forward 'conj': {len(dk5_forward)}")
print("=" * 80)

for sent_id, sentence, token, newpart in dk5_forward:
    
    print(f"SENTENCE ID: {sent_id}")
    print(f"Token: {token.form} (ID={token.id}) → Head ID={token.head}")
    print(f"Sentence-level Newpart: {newpart if newpart else 'UNKNOWN'}")
    
    # Safe sentence text
    text = sentence.metadata.get("text")
    if not text:
        text = " ".join(t.form for t in sentence.get_tokens() if t.form)
    
    print(f"Text: {text}")
    print("-" * 80)

Total DK5 forward 'conj': 47
SENTENCE ID: Dk5-B s8
Token: hufraward (ID=16) → Head ID=17
Sentence-level Newpart: Dk5 1
Text: _ _ _ abar čiyōn padīrēnd paygāmbarān ī pēš az zarduxšt ēn dēn ud čiyōn āmad ī abēzag hufraward zarduxšt ī spitāmān , ,
--------------------------------------------------------------------------------
SENTENCE ID: Dk5-B s15
Token: kē (ID=1) → Head ID=17
Sentence-level Newpart: Dk5 2
Text: _ _ _ _ kē margīh _ kē agārīh _ ud tā =z kē andardastōš ud abārīg armēštīh wēnābdāg hanǰaman paydāgīhā abar mad ud tā =z gurgān abārīg dadān kē =šān az ān ī pōrušāsp xwēšāwandān ī ǰādūg dēwēzag čārīhxwāhīh bē pad =iz uzmāyišn frāz awi =šān abgand
--------------------------------------------------------------------------------
SENTENCE ID: Dk5-B s15
Token: kē (ID=3) → Head ID=17
Sentence-level Newpart: Dk5 2
Text: _ _ _ _ kē margīh _ kē agārīh _ ud tā =z kē andardastōš ud abārīg armēštīh wēnābdāg hanǰaman paydāgīhā abar mad ud tā =z gurgān abārīg dadān kē =šān az ān ī pōrušāsp xw

In [26]:
# ============================================================
# DK5 forward-headed 'conj' tokens with propagated newpart
# Output to a TXT file
# ============================================================

output_file = "dk5_forward_conj.txt"

dk5_forward = []

# Open the file in write mode
with open(output_file, "w", encoding="utf-8") as f:

    for sentence in syntactically_parsed_corpus:
        
        raw_sid = sentence.metadata.get("SENTENCE ID", "")
        clean_sid = raw_sid.split("\t")[0].strip()
        
        # Filter only DK5 sentences (case-insensitive)
        if not clean_sid.lower().startswith("dk5"):
            continue
        
        last_newpart = None  # keep track of last seen newpart
        
        for token in sentence.get_tokens():
            
            # Update last_newpart if current token has it
            token_newpart = token.misc.get("newpart")
            if token_newpart:
                last_newpart = token_newpart
            
            # Only forward-headed 'conj'
            if (
                token.deprel == "conj"
                and str(token.id).isdigit()
                and str(token.head).isdigit()
                and int(token.head) > int(token.id)
            ):
                dk5_forward.append((clean_sid, sentence, token, last_newpart))
                
                # Write to file
                f.write("=" * 80 + "\n")
                f.write(f"SENTENCE ID: {clean_sid}\n")
                f.write(f"Token: {token.form} (ID={token.id}) → Head ID={token.head}\n")
                f.write(f"Sentence-level Newpart: {last_newpart if last_newpart else 'UNKNOWN'}\n")
                
                # Get sentence text safely
                text = sentence.metadata.get("text")
                if not text:
                    text = " ".join(t.form for t in sentence.get_tokens() if t.form)
                f.write(f"Text: {text}\n")
    f.write("=" * 80 + "\n")
    f.write(f"Total DK5 forward 'conj' tokens: {len(dk5_forward)}\n")

print(f"Results written to {output_file}")

Results written to dk5_forward_conj.txt


In [20]:
for sentence in syntactically_parsed_corpus[100:105]:
    print(sentence.metadata)
    print("KEYS:", sentence.metadata.keys())
    print("-" * 60)

{'sent_id': '32', 'text': '_ _ _ _ ud mādayān ud rāstīh pēšēnīg ud dastwarān gōwišn ān čē pad čim andar pāyišn _ ēd čē pad rōšngarīh ī wizīr , nibišt', 'SENTENCE ID': 'DD-K35 s32\t_\t_\t_\t_\t_\t_\t_\t_\t_', 'TRANSLATION': '[eng] [JD] But I have written in the answer the essence of the truth from the reasonable statements of the early Dastwars and whatever was useful to illustrate their judgments. -\t_\t_\t_\t_\t_\t_\t_\t_\t_', 'COMMENT': 'This may actually continue the preceding sentence, which would then include a parenthesis.\t_\t_\t_\t_\t_\t_\t_\t_\t_', 'genre': 'unknown'}
KEYS: dict_keys(['sent_id', 'text', 'SENTENCE ID', 'TRANSLATION', 'COMMENT', 'genre'])
------------------------------------------------------------
{'sent_id': '33', 'text': '_ _ _ _ agar ēdōn ī čim rāy bowandag _ ayāb wizīr rāy rōšn nē wēnīhēd , nē pargast az abowandagīh ī dēn wizīr pad rōšnīhānimūdārīh ud druyistčimīh bē az abowandag madārīh ī amāh ō šnāyišn ī ān dēn X nigēzag pargūd az ōšīh ī ān =iz ī =mān xwā

In [22]:
for sentence in syntactically_parsed_corpus[:1]:
    for token in sentence.get_tokens():
        print(token)
        print(token.feats)
        print(token.misc)
        break

{'Mood': 'Ind', 'Number': 'Sing', 'Person': '3', 'Subcat': 'Tran', 'Tense': 'Past', 'VerbForm': 'Fin'}
{'transliteration': '[p]wrsyt', 'folionew': 'K20_143r', 'line': '4.0', 'newpart': 'AOD 1', 'AOD 1.1': '', 'avestan': '–', 'meaning': 'ask', 'uncertain': "['L']", 'comment': 'read:pwrsyt', 'token_lang': 'pal', 'meaning_lang': 'eng', 'lemma_langs': 'pal'}
